# 03 — Backtest & Report

Turns a stage-02 out-of-sample **signal** into a traded index-timing strategy and
reports it against buy-and-hold.

- **Position**: long/short/flat from the signal, **vol-targeted** to a constant annual
  risk (capped leverage), refreshed every H trading days and held in between.
- **Costs**: `cost_bps` per turnover (default 10 bps), optional quarterly-roll cost.
- **Accounting**: position applied to *next-day* index returns (no same-day
  look-ahead) -> daily equity curve.

Outputs: performance summary, equity curve, drawdown, monthly-return heatmap (color
fixed to +/-10%), and yearly-return bars.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.config import load_config
from src.backtest import run_backtest, buy_and_hold
from src.perf import performance_summary
from src.viz import (plot_equity, plot_drawdown, plot_monthly_heatmap,
                     plot_yearly_bars, monthly_return_table)
from index_ticker import SP500

cfg = load_config(PROJECT_ROOT / "config.yaml")
bt_cfg = cfg.backtest

def _require(path, name):
    if not Path(path).exists():
        raise FileNotFoundError(f"{name} not found at {path}.\nRun the earlier notebooks first.")
    return path

signals = pd.read_parquet(_require(cfg.signals_path, "oos_signals"))
metrics = pd.read_csv(_require(cfg.metrics_path, "walkforward_metrics"))
panel = pd.read_parquet(_require(cfg.indices_path, "indices_raw"))
close = panel[("Close", SP500)]
print("signals:", [c for c in signals.columns if not c.startswith('realized__')])

## 1 — Pick a signal and run the backtest

By default the signal with the best non-overlapping Rank IC is chosen; set
`backtest.signal` in `config.yaml` (e.g. `histgbm_clf__h21`) to fix one.

In [ ]:
# choose signal column
if bt_cfg.get('signal'):
    signal_col = bt_cfg['signal']
else:
    best = metrics.sort_values('rank_ic_nonoverlap', ascending=False).iloc[0]
    H_best = cfg.horizons[best['horizon']]
    signal_col = f"{best['model']}__h{H_best}"

H = int(signal_col.split('__h')[1])
sig = signals[signal_col].dropna()
daily_ret = close.pct_change().reindex(sig.index)

bt = run_backtest(
    sig, daily_ret, H=H,
    cost_bps=bt_cfg.get('cost_bps', 20),
    roll=bt_cfg.get('roll', 'quarterly'),
    roll_cost_bps=bt_cfg.get('roll_cost_bps', 0),
    positioning=bt_cfg.get('positioning', 'long_short_flat'),
    sizing=bt_cfg.get('sizing', 'vol_target'),
    target_vol=bt_cfg.get('target_vol', 0.10),
    max_leverage=bt_cfg.get('max_leverage', 3.0),
    deadband=bt_cfg.get('deadband', 0.0),
    vol_lookback=bt_cfg.get('vol_lookback', 20),
    conviction=bt_cfg.get('conviction', False),
)
bench = buy_and_hold(daily_ret, cost_bps=bt_cfg.get('cost_bps', 20))

print(f"signal      : {signal_col}  (H={H})")
print(f"span        : {sig.index.min().date()} -> {sig.index.max().date()}  ({len(sig)} days)")
print(f"positioning : {bt_cfg.get('positioning','long_short_flat')} | "
      f"sizing: {bt_cfg.get('sizing','vol_target')} @ {bt_cfg.get('target_vol',0.10):.0%} | "
      f"cost: {bt_cfg.get('cost_bps',20)} bps")

## 2 — Performance summary

In [ ]:
summary = pd.DataFrame({
    'strategy': performance_summary(bt['strat_ret'], turnover=bt['turnover'],
                                    position=bt['position']),
    'buy_hold': performance_summary(bench['strat_ret']),
})
pct_rows = ['total_return', 'cagr', 'ann_vol', 'max_drawdown', 'hit_rate', 'avg_gross_exposure']
fmt = summary.copy()
for r in fmt.index:
    if r in pct_rows:
        fmt.loc[r] = summary.loc[r].map(lambda v: f'{v:.2%}' if pd.notna(v) else '')
    else:
        fmt.loc[r] = summary.loc[r].map(lambda v: f'{v:.2f}' if pd.notna(v) else '')
fmt

## 3 — Equity curve & drawdown

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7),
                               gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
plot_equity(bt['strat_equity'], bench['strat_equity'], ax=ax1)
plot_drawdown(bt['strat_equity'], ax=ax2)
plt.tight_layout()
plt.show()

## 4 — Monthly returns (heatmap, fixed +/-10% scale)

In [ ]:
ax = plot_monthly_heatmap(bt['strat_ret'], vmin=-0.10, vmax=0.10)
plt.tight_layout()
plt.show()

## 5 — Yearly returns

In [ ]:
ax = plot_yearly_bars(bt['strat_ret'])
plt.tight_layout()
plt.show()

---
### Notes & honest caveats
- Returns are in % of equity, so ES vs MES is cosmetic; leverage is implied by the
  vol target and capped by `max_leverage`.
- 10 bps/turn is moderate-to-conservative for liquid index futures (real all-in is often a few
  bps); it stays configurable. Turnover here is low (rebalance every H days).
- The signal is genuinely out-of-sample (expanding walk-forward with an H-day
  embargo, per-fold bucketing), but this is still a single historical path on one
  index. Treat the numbers as indicative, not a live-trading promise; a flat or
  weak result is a real finding, not a bug.